# Advance Crime Data Pipeline

## Stage 4: Aggregation Layer — Gold

**Stakeholder:** Police Force Analytics Unit <br>
**Stage:** 4 of 4: Aggregation <br>
**Medallion Layer:** Gold 🥇 <br>
**Police Forces:** West Midlands · Thames Valley · Surrey · Cumbria <br>
**Authors:** Group 1 <br>
**Last Updated:** 26 May 2026

This notebook aggregates the enriched Silver dataset to a single, consistent reporting grain for BI consumption. All aggregation decisions and metric calculations are documented below. The output is a clean, BI-ready dataset ready for Power BI.

---

### Purpose

This notebook forms the **Gold layer** of the medallion pipeline. It aggregates enriched crime records to the reporting grain and calculates normalised metrics, enabling in-depth analysis of the data.

---

### Reporting Grain

**One row = one police force × one district × one year × one month × one crime type**


---

# Data Dictionary — Gold Reporting Dataset

**Table:** `CRIME_PIPELINE.REPORTING.GOLD_CRIME_REPORTING`  
**Grain:** One row = one police force × one district × one year × one month × one crime type  

---

| Column | Description | Type | Grain | Source | Calculations & Assumptions |
|---|---|---|---|---|---|
| `force_name` | Name of the police force | Text | One value per force | UK Police Crime Data (data.police.uk) | No transformation applied |
| `district_name` | Name of the local authority district | Text | One value per district within each force | UK Police Crime Data (data.police.uk) | Old Cumbria districts retained as per original crime data |
| `year` | Calendar year of the crime record | Integer | One value per year (2023–2026) | Derived from `month` column in source data | Extracted from date field in Python |
| `month_num` | Month number of the crime record | Integer | 1–12 representing January to December | Derived from `month` column in source data | Extracted from date field in Python |
| `crime_type` | Category of crime | Text | 14 distinct crime categories | UK Police Crime Data (data.police.uk) | Standardised to Title Case in Python |
| `crime_count` | Total number of crimes recorded | Integer | Aggregated count per force × district × year × month × crime type | Derived from individual crime records in source data | COUNT of individual crime records grouped at reporting grain |
| `latitude` | Central latitude coordinate of the district | Float | One value per district | Manually sourced and averaged from LSOA coordinates | Represents district centroid, not individual crime location |
| `longitude` | Central longitude coordinate of the district | Float | One value per district | Manually sourced and averaged from LSOA coordinates | Represents district centroid, not individual crime location |
| `population_2024` | Estimated resident population of the district | Integer | One value per district | ONS Mid-Year Population Estimates 2024 (Nomis) | Cumbria districts split proportionally from Cumberland and Westmorland & Furness 2024 figures |
| `median_house_price_2024` | Median residential property sale price | Integer | One value per district | ONS/Land Registry House Price Statistics (Dec 2024) | Cumbria and merged Thames Valley districts inherited from parent authority |
| `imd_rank_2024` | Index of Multiple Deprivation overall rank | Integer | One value per district (rank out of 296) | Ministry of Housing, Communities & Local Government IoD 2024 | Lower rank = more deprived. Cumbria districts use Cumberland/Westmorland rank |
| `income_rank_2024` | Income deprivation domain rank | Integer | One value per district (rank out of 296) | Ministry of Housing, Communities & Local Government IoD 2024 | Lower rank = more deprived |
| `employment_rank_2024` | Employment deprivation domain rank | Integer | One value per district (rank out of 296) | Ministry of Housing, Communities & Local Government IoD 2024 | Lower rank = more deprived |
| `health_rank_2024` | Health deprivation and disability domain rank | Integer | One value per district (rank out of 296) | Ministry of Housing, Communities & Local Government IoD 2024 | Lower rank = more deprived |
| `crime_rank_2024` | Crime deprivation domain rank | Integer | One value per district (rank out of 296) | Ministry of Housing, Communities & Local Government IoD 2024 | Lower rank = more deprived. Note: this is a deprivation-based crime score, not actual crime count |
| `crime_rate_per_1000` | Crimes per 1,000 residents at district level | Float | One value per force × district × year × month × crime type | Derived from `crime_count` and `population_2024` | `crime_count / population_2024 × 1,000` |

---


### How to Use

**Prerequisites:** `CRIME_PIPELINE.CLEAN.SILVER_CRIME_ENRICHED` must be populated. Run `03_feature_engineering.ipynb` first if not already executed.

**Power BI:** Connect directly to `CRIME_PIPELINE.REPORTING.GOLD_CRIME_REPORTING` using the Snowflake connector in Power BI Desktop. The 

The data is also outputed as a zipped CSV.

## 1. Environment Setup

This section sets up the Snowflake and Python environment required for the pipeline. It creates the necessary **warehouse**, **database**, and **schemas** if they do not already exist.

In [ ]:
%%sql -r dataframe_1
USE ROLE SYSADMIN;

In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.connector.pandas_tools import write_pandas

session = get_active_session()
session.sql("USE DATABASE CRIME_PIPELINE").collect()
session.sql("USE SCHEMA REPORTING").collect()

# Source and destination table references
ENRICHED_TABLE = "CRIME_PIPELINE.CLEAN.SILVER_CRIME_ENRICHED"
GOLD_TABLE     = "GOLD_CRIME_REPORTING"

# Confirm session is pointing at the correct database and schema
print("Session database :", session.get_current_database())
print("Session schema   :", session.get_current_schema())

## 2. Read from Enriched Silver Table and Initial Inspection

Load the fully enriched dataset from `SILVER_CRIME_ENRICHED`. This is the hand-off from the feature engineering layer — no further cleaning is applied here.

In [ ]:
# Read enriched Silver table
crime = session.table(ENRICHED_TABLE).to_pandas()

# Standardise column names: strip whitespace, lowercase, replace spaces with underscores
crime.columns = [c.strip().lower().replace(" ", "_") for c in crime.columns]

# Filter to March 2024 onwards -- two years of data
crime["month"] = pd.to_datetime(crime["month"])
crime = crime[crime["month"] >= "2023-01-01"]

# Record baseline count -- used in reconciliation report
baseline_count = len(crime)

print("Enriched Silver rows :", len(crime))
print("Date range           :", crime["month"].min(), "→", crime["month"].max())
print("Columns              :", crime.columns.tolist())

In [ ]:
# Visual inspection of the first five rows to confirm structure
crime.head()

## 3. Pre-Aggregation Preparation

In this step, pipeline provenance columns that carry no analytical value are removed before aggregation. Exposing internal tracking fields to BI consumers would add noise to the reporting dataset.

The following columns are dropped before aggregation: <br>
1. **source_file** — pipeline provenance, not required for reporting <br>
2. **source_month** — duplicate of `month`, not required for reporting <br>
3. **falls_within** — replaced by the cleaner `force_name` column <br>
4. **lsoa_code** — individual record level, not meaningful after district aggregation <br>
5. **lsoa_name** — replaced by `district_name` derived in the feature engineering layer <br>
6. **crime_id** — individual record identifier, not meaningful after aggregation <br>
7. **last_outcome_category** — outcome data not included in reporting grain <br>
8. **longitude / latitude (record level)** — replaced by district centroid coordinates from enrichment data

In [ ]:
cols_to_drop = [
    "source_file",
    "source_month",
    "falls_within",
    "lsoa_code",
    "lsoa_name",
    "crime_id",
    "last_outcome_category",
]

# errors='ignore' ensures the cell is idempotent -- safe to re-run even if columns were already dropped
crime = crime.drop(columns=cols_to_drop, errors="ignore")

print("Columns dropped  :", cols_to_drop)
print("Remaining columns:", crime.columns.tolist())

In [ ]:
# Drop suppressed location records -- district unknown, cannot be mapped
crime = crime[crime["district_name"] != "NOT_RECORDED"]

print(f"Rows after NOT_RECORDED filter : {len(crime):,}")

# Reset baseline to post-filter count -- reconciliation compares against this
baseline_count = len(crime)
print(f"Baseline count set             : {baseline_count:,}")

## 4. Aggregate to Reporting Grain

In this step, individual crime records are aggregated to the reporting grain: `force_name × district_name × year × month_num × crime_type`.

Each metric is calculated as follows: <br>
- **`crime_count`** — count of all crime records at the grain <br>
- **`latitude` / `longitude`** — district centroid coordinates (first value — same for all records in a district) <br>
- **`population_2024`** — district population (first value — one per district) <br>
- **`median_house_price_2024`** — median house price (first value — one per district) <br>
- **`imd_rank_2024`** and other deprivation ranks — first value per district

In [ ]:
# Define reporting grain
GRAIN_COLS = ["force_name", "district_name", "year", "month_num", "crime_type"]

# Aggregate all metrics at reporting grain
gold = crime.groupby(GRAIN_COLS, as_index=False).agg(
    crime_count              = ("district_name",          "count"),
    latitude                 = ("latitude",               "first"),
    longitude                = ("longitude",              "first"),
    population_2024          = ("population_2024",        "first"),
    median_house_price_2024  = ("median_house_price_2024", "first"),
    imd_rank_2024            = ("imd_rank_2024",          "first"),
    income_rank_2024         = ("income_rank_2024",       "first"),
    employment_rank_2024     = ("employment_rank_2024",   "first"),
    health_rank_2024         = ("health_rank_2024",       "first"),
    crime_rank_2024          = ("crime_rank_2024",        "first"),
)

# Round float columns for cleaner BI display
float_cols = [c for c in gold.columns if gold[c].dtype == "float64"]
gold[float_cols] = gold[float_cols].round(2)

print(f"Gold rows : {len(gold):,}")
print(f"Columns   : {gold.columns.tolist()}")

## 5. Calculate Normalised Crime Rate

Calculate `crime_rate_per_1000` — the number of crimes per 1,000 residents at district level. This enables fair comparison between districts of different population sizes.

**Formula:** `crime_count / population_2024 × 1,000` <br>

Districts with null population (suppressed location records with `NOT_RECORDED` district) will produce a null rate — this is expected and documented.

In [ ]:
# Calculate crime rate per 1,000 residents at district level
# Null population produces null rate -- expected for NOT_RECORDED district records
gold["crime_rate_per_1000"] = (
    gold["crime_count"] / gold["population_2024"] * 1000
).round(2)

print("Null crime rates:", gold["crime_rate_per_1000"].isnull().sum())
print(gold[["force_name", "district_name", "crime_type",
            "crime_count", "population_2024", "crime_rate_per_1000"]].head(10))

## 6. Reorder Columns

Reorder columns to match the target output format for Power BI.

In [ ]:
# Reorder columns to match target output format
gold = gold[[
    "force_name",
    "district_name",
    "year",
    "month_num",
    "crime_type",
    "crime_count",
    "latitude",
    "longitude",
    "population_2024",
    "median_house_price_2024",
    "imd_rank_2024",
    "income_rank_2024",
    "employment_rank_2024",
    "health_rank_2024",
    "crime_rank_2024",
    "crime_rate_per_1000"
]]

print("Final columns:", gold.columns.tolist())
print(gold.head(5))

## 7. Gold Validation Report

Confirm the Gold dataset is BI-ready before export. Three checks are performed: <br>
- No duplicate rows at the reporting grain <br>
- No missing values in required reporting fields <br>
- Total crime count matches the Silver baseline — confirms no records were lost during aggregation

In [ ]:
print("╔══════════════════════════════════════════════════════════════╗")
print("║              GOLD VALIDATION REPORT                         ║")
print("╚══════════════════════════════════════════════════════════════╝")

# Overview stats
print(f"Gold rows              : {len(gold):,}")
print(f"Distinct forces        : {gold['force_name'].nunique()}")
print(f"Distinct districts     : {gold['district_name'].nunique()}")
print(f"Distinct years         : {sorted(gold['year'].unique().tolist())}")
print(f"Distinct crime types   : {gold['crime_type'].nunique()}")

# 1. Duplicate check at reporting grain
dup_count = gold.duplicated(subset=GRAIN_COLS).sum()
print(f"\nDuplicates at grain    : {dup_count}")
assert dup_count == 0, "Duplicate rows found at reporting grain — investigate before export"

# 2. Crime count reconciliation -- Gold sum should equal Silver baseline row count
total_crimes = gold["crime_count"].sum()
print(f"\nTotal crimes (Gold)    : {total_crimes:,}")
print(f"Baseline rows (Silver) : {baseline_count:,}")
assert total_crimes == baseline_count, \
    f"Crime count mismatch: {total_crimes} vs {baseline_count}"
print("Crime count reconciliation: ✓ PASSED")

# 3. Null check on required reporting fields
required_cols = ["force_name", "district_name", "year", "month_num", "crime_type", "crime_count"]
null_check = gold[required_cols].isnull().sum()
print(f"\nNull check on required fields:")
print(null_check)
assert null_check.sum() == 0, "Null values found in required reporting fields"

## 8. Inspect Gold Dataset

Final visual inspection and high level statistics before export.

In [ ]:
print("Shape  :", gold.shape)
print("Columns:", gold.columns.tolist())

gold.head(10)

In [ ]:
# High level statistics -- sense check before export
print("── Crime count by force ──")
print(gold.groupby("force_name")["crime_count"].sum().sort_values(ascending=False))

print("\n── Crime count by crime type ──")
print(gold.groupby("crime_type")["crime_count"].sum().sort_values(ascending=False))

print("\n── Crime count by year ──")
print(gold.groupby("year")["crime_count"].sum().sort_index())

## 9. Export to Gold Table

Persist the aggregated reporting dataset to `CRIME_PIPELINE.REPORTING.GOLD_CRIME_REPORTING`. This is the final output of the pipeline — the table Power BI connects to directly.

In [ ]:
# Reset index before writing to suppress non-standard index warning
gold = gold.reset_index(drop=True)

success, nchunks, nrows, _ = write_pandas(
    conn=session.connection,
    df=gold,
    table_name=GOLD_TABLE,
    database="CRIME_PIPELINE",
    schema="REPORTING",
    auto_create_table=True,  # creates table if it does not exist
    overwrite=True           # replaces existing data on each run
)

# Verify written row count matches Gold row count
written_count = session.table(f"CRIME_PIPELINE.REPORTING.{GOLD_TABLE}").count()
assert written_count == len(gold), \
    f"Row count mismatch: {written_count} written vs {len(gold)} expected"

print(f"Gold table written successfully")
print(f"Table  : CRIME_PIPELINE.REPORTING.{GOLD_TABLE}")
print(f"Rows   : {written_count:,}")

## 10. Export to CSV

Export the Gold reporting dataset as a compressed CSV for local use or direct import into Power BI.

In [ ]:
gold.to_csv("gold_crime_reporting.csv.gz", 
            index=False, compression="gzip")

print("Exported : gold_crime_reporting.csv.gz")
print(f"Rows     : {len(gold):,}")
print(f"Columns  : {len(gold.columns)}")
print(f"\nColumn list: {gold.columns.tolist()}")

In [ ]:
print("── Rows per force ──")
print(gold.groupby("force_name").size().sort_values(ascending=False))

print("\n── Rows per district (top 30) ──")
print(gold.groupby("district_name").size().sort_values(ascending=False).head(30))

print("\n── Rows per year ──")
print(gold.groupby("year").size().sort_values(ascending=False))

print(  "\n── Rows per crime type ──")
print(gold.groupby("crime_type").size().sort_values(ascending=False))

print("\n── Total distinct grain combinations ──")
print(f"Forces     : {gold['force_name'].nunique()}")
print(f"Districts  : {gold['district_name'].nunique()}")
print(f"Years      : {gold['year'].nunique()}")
print(f"Months     : {gold['month_num'].nunique()}")
print(f"Crime types: {gold['crime_type'].nunique()}")